In this notebook, I will use the data generated in poseRecognition.ipynb to extract additional features. The primary focus will be on calculating the angles between joints and measuring the speed of movement.

In [ ]:
pip install pandas

In [ ]:
import pandas as pd


1. Load the existing data

In [ ]:
data = pd.read_csv('output\poseDatasied1.csv')
data 

1.2 Data cleanup 

Since the data was generated using MediaPipe, some joints are not relevant for my analysis. The joints are organized as follows:
| ID  | Landmark           |
|-----|------------------|
| 0   | NOSE              |
| 1   | LEFT_EYE_INNER    |
| 2   | LEFT_EYE          |
| 3   | LEFT_EYE_OUTER    |
| 4   | RIGHT_EYE_INNER   |
| 5   | RIGHT_EYE         |
| 6   | RIGHT_EYE_OUTER   |
| 7   | LEFT_EAR          |
| 8   | RIGHT_EAR         |
| 9   | MOUTH_LEFT        |
| 10  | MOUTH_RIGHT       |
| 11  | LEFT_SHOULDER     |
| 12  | RIGHT_SHOULDER    |
| 13  | LEFT_ELBOW        |
| 14  | RIGHT_ELBOW       |
| 15  | LEFT_WRIST        |
| 16  | RIGHT_WRIST       |
| 17  | LEFT_PINKY        |
| 18  | RIGHT_PINKY       |
| 19  | LEFT_INDEX        |
| 20  | RIGHT_INDEX       |
| 21  | LEFT_THUMB        |
| 22  | RIGHT_THUMB       |
| 23  | LEFT_HIP          |
| 24  | RIGHT_HIP         |
| 25  | LEFT_KNEE         |
| 26  | RIGHT_KNEE        |
| 27  | LEFT_ANKLE        |
| 28  | RIGHT_ANKLE       |
| 29  | LEFT_HEEL         |
| 30  | RIGHT_HEEL        |
| 31  | LEFT_FOOT_INDEX   |
| 32  | RIGHT_FOOT_INDEX  |


I will exclude irrelevant joints, such as ears ore pinkys, from the analysis.


In [ ]:
needed = (
    (11, "LEFT_SHOULDER"),
    (12, "RIGHT_SHOULDER"),
    (13, "LEFT_ELBOW"),
    (14, "RIGHT_ELBOW"),
    (15, "LEFT_WRIST"),
    (16, "RIGHT_WRIST"),
    (23, "LEFT_HIP"),
    (24, "RIGHT_HIP"),
    (25, "LEFT_KNEE"),
    (26, "RIGHT_KNEE"),
    (27, "LEFT_ANKLE"),
    (28, "RIGHT_ANKLE"),
    (29, "LEFT_HEEL"),
    (30, "RIGHT_HEEL"),
    (31, "LEFT_FOOT_INDEX"),
    (32, "RIGHT_FOOT_INDEX")
)


In [ ]:
needed_ids = [id for id, name in needed]

data_filtered = data[data["joint"].isin(needed_ids)]
data_filtered


2. Compute joint angles


In [ ]:
import numpy as np

Pivot data to have joints as columns per frame
Columns will be like: joint0_x, joint0_y, joint0_z, joint1_x, ...

In [ ]:

pose_pivot = data.pivot(index='frame', columns='joint', values=['x','y','z'])
pose_pivot.columns = [f"{axis}{joint}" for axis, joint in pose_pivot.columns]


Calculate the angle at point b formed by points a-b-c in 3D.
Returns angle in degrees.


In [ ]:
def calculate_angle(a, b, c):

    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cos_angle = np.clip(cos_angle, -1.0, 1.0)  # Numerical stability
    angle = np.arccos(cos_angle)
    return np.degrees(angle)


compute elbow angle (shoulder-elbow-wrist)

In [ ]:
angles = []
for idx, row in pose_pivot.iterrows():
    shoulder = (row['x0'], row['y0'], row['z0'])  
    elbow = (row['x1'], row['y1'], row['z1'])
    wrist = (row['x2'], row['y2'], row['z2'])
    angle = calculate_angle(shoulder, elbow, wrist)
    angles.append(angle)

pose_pivot['elbow_angle'] = angles

pose_pivot
